# P4 · Cold-start для плотной per-pair памяти (M-only)

Метод из `p3_perpair_memory.ipynb`: `M_{u,i}(t)=Σ F̂_train(w)·exp(-α(t-t_e))`, предсказание = ранжирование по строке `M[u,:]`.
**Проблема:** холодный юзер → строка `M[u,:]` разрежена/нулевая → слабое предсказание (P2b: на холодных headroom велик).

**Рычаг:** асимметрия `items ≪ users` — item'ы малочисленны и частые ⇒ их статистики ВСЕГДА «тёплые». Идея — достроить
разреженную строку холодного юзера через тёплую **item-side** структуру.

**Жёсткое ограничение:** ОДНА трансформация ОДНОГО состояния (напр. `M·A`, `A` frozen-from-train), **не ансамбль**
(нельзя смешивать `M[u,:]` с популярностью/вторым предсказателем на выходе). Статистики — только из train (каузально).

**План:** (1) диагностика — per-warmth NDCG для M-only (где именно проседает); (2) single-model митигации (item-item
co-occurrence сглаживание и др. — допишет панель `coldstart-dense-memory`); (3) проверка: подъём холодных без вреда тёплым.

In [2]:
import numpy as np, polars as pl, plotly.express as px, timeit, sys
from sklearn.metrics import ndcg_score
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from tgb.nodeproppred.evaluate import Evaluator
from torch_geometric.loader import TemporalDataLoader
REPO = "/Users/aleksandrpanysev/Documents/GitHub/2Q_2026_tgn_user_item"
if REPO not in sys.path: sys.path.insert(0, REPO)

def load_dataset(name, bs=200):
    ds = PyGNodePropPredDataset(name=name, root="datasets")
    data = ds.get_TemporalData()
    tr, va, te = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    return dict(ds=ds, data=data, num_classes=ds.num_classes, num_nodes=data.num_nodes,
                evaluator=Evaluator(name=name), eval_metric=ds.eval_metric,
                loaders={s: TemporalDataLoader(d, batch_size=bs) for s, d in [("train", tr), ("val", va), ("test", te)]})

def make_rank_phi(DS):  # rank/ECDF по train (каузально)
    tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    sw = np.sort(tr.msg[:, 0].numpy()); n = len(sw)
    return lambda w: np.searchsorted(sw, w, side="right") / n

class HawkesMemory:
    def __init__(self, N, C, alpha, phi):
        self.C = C; self.M = np.zeros((N, C)); self.tlast = np.zeros((N, C)); self.alpha = float(alpha); self.phi = phi
    def update(self, src, dst, t, w):
        if src.size == 0: return
        idx = src.astype(np.int64) * self.C + dst.astype(np.int64); tb = float(t.max())
        uniq, inv = np.unique(idx, return_inverse=True); add = np.bincount(inv, weights=self.phi(w), minlength=uniq.size)
        fM, fT = self.M.reshape(-1), self.tlast.reshape(-1)
        fM[uniq] = (fM[uniq] * np.exp(-self.alpha * np.clip(tb - fT[uniq], 0, None)) + add) if self.alpha > 0 else fM[uniq] + add
        fT[uniq] = tb
    def read(self, users, t_read):
        r = self.M[users]
        return r * np.exp(-self.alpha * np.clip(t_read - self.tlast[users], 0, None)) if self.alpha > 0 else r

DS = load_dataset("tgbn-genre")
DCHAR = 86400.0; KAPPA = 25
print(f"genre: nodes={DS['num_nodes']} classes={DS['num_classes']} edges={DS['data'].src.numel()}")

genre: nodes=1505 classes=513 edges=17858395


In [3]:
def stream_cohort(DS, mem, cnt, loader, strict=True, collect=False):
    ds = DS["ds"]; label_t = ds.get_label_time(); recs = []
    for batch in loader:
        sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
        if float(batch.t[-1]) > label_t:
            lt = ds.get_node_label(batch.t[-1])
            if lt is None: break
            lab_ts0 = float(lt[0][0]); ls, labs = lt[1].numpy(), lt[2].numpy()
            label_t = ds.get_label_time(); pm = tn < (lab_ts0 if strict else label_t)
            mem.update(sn[pm], dn[pm], tn[pm], wn[pm]); np.add.at(cnt, sn[pm], 1)
            if collect:
                pred = mem.read(ls, lab_ts0)
                for i in range(len(ls)):
                    if labs[i].sum() > 0:
                        recs.append((int(cnt[ls[i]]), float(ndcg_score(labs[i:i+1], pred[i:i+1], k=10))))
            sn, dn, tn, wn = sn[~pm], dn[~pm], tn[~pm], wn[~pm]
        mem.update(sn, dn, tn, wn); np.add.at(cnt, sn, 1)
    return recs

def run_cohort(DS, phi, alpha):
    mem = HawkesMemory(DS["num_nodes"], DS["num_classes"], alpha, phi); cnt = np.zeros(DS["num_nodes"])
    stream_cohort(DS, mem, cnt, DS["loaders"]["train"]); stream_cohort(DS, mem, cnt, DS["loaders"]["val"])
    recs = stream_cohort(DS, mem, cnt, DS["loaders"]["test"], collect=True)
    DS["ds"].reset_label_time()
    return pl.DataFrame(recs, schema=["warmth", "ndcg"], orient="row")

rphi = make_rank_phi(DS)
alpha = np.log(2) / (KAPPA * DCHAR)
co = run_cohort(DS, rphi, alpha)
print(f"genre test (user,день): {co.height} | overall NDCG@10 = {co['ndcg'].mean():.4f}")
tab = (co.with_columns(pl.col("warmth").qcut(5, allow_duplicates=True).alias("bin"))
         .group_by("bin").agg(pl.col("warmth").median().alias("med_warmth"),
                              pl.col("ndcg").mean().round(4).alias("ndcg"), pl.len().alias("n"))
         .sort("med_warmth"))
print("per-warmth когорты (холодные → тёплые):"); print(tab)

genre test (user,день): 32151 | overall NDCG@10 = 0.5203
per-warmth когорты (холодные → тёплые):
shape: (5, 4)
┌────────────────┬────────────┬────────┬──────┐
│ bin            ┆ med_warmth ┆ ndcg   ┆ n    │
│ ---            ┆ ---        ┆ ---    ┆ ---  │
│ cat            ┆ f64        ┆ f64    ┆ u32  │
╞════════════════╪════════════╪════════╪══════╡
│ (-inf, 6715]   ┆ 3259.0     ┆ 0.4862 ┆ 6431 │
│ (6715, 15388]  ┆ 10596.0    ┆ 0.4779 ┆ 6430 │
│ (15388, 26316] ┆ 20438.0    ┆ 0.5074 ┆ 6431 │
│ (26316, 45969] ┆ 35218.0    ┆ 0.5274 ┆ 6429 │
│ (45969, inf]   ┆ 69495.0    ┆ 0.6027 ┆ 6430 │
└────────────────┴────────────┴────────┴──────┘


In [4]:
fig = px.line(tab.with_columns(pl.col("med_warmth").cast(pl.Int64).cast(pl.Utf8)).to_pandas(),
              x="med_warmth", y="ndcg", markers=True,
              title="genre M-only: NDCG@10 vs warmth (накопл. взаимодействий)",
              labels={"med_warmth": "медианная warmth когорты", "ndcg": "NDCG@10"})
fig.show()

print("Истинно холодный хвост (genre test):")
for thr in [10, 50, 200, 1000, 5000]:
    sub = co.filter(pl.col("warmth") < thr)
    if sub.height:
        print(f"  warmth<{thr:>5}: n={sub.height:>6} ({sub.height/co.height*100:4.1f}%)  NDCG@10={sub['ndcg'].mean():.4f}")
    else:
        print(f"  warmth<{thr:>5}: 0")
print(f"  warmth==0 (нет истории): n={co.filter(pl.col('warmth')==0).height}")

Истинно холодный хвост (genre test):
  warmth<   10: n=    18 ( 0.1%)  NDCG@10=0.4036
  warmth<   50: n=    74 ( 0.2%)  NDCG@10=0.4197
  warmth<  200: n=   184 ( 0.6%)  NDCG@10=0.4592
  warmth< 1000: n=   892 ( 2.8%)  NDCG@10=0.4644
  warmth< 5000: n=  4783 (14.9%)  NDCG@10=0.4827
  warmth==0 (нет истории): n=0


In [5]:
def cold_report(name):
    D = load_dataset(name)
    phi = make_rank_phi(D); a = np.log(2) / (KAPPA * 86400.0)
    c = run_cohort(D, phi, a)
    print(f"\n=== {name} ===  test (user,день)={c.height} | overall NDCG@10={c['ndcg'].mean():.4f}")
    print(f"  warmth==0 (НЕТ истории): n={c.filter(pl.col('warmth')==0).height}")
    for thr in [10, 50, 200, 1000]:
        s = c.filter(pl.col("warmth") < thr)
        if s.height:
            print(f"  warmth<{thr:>5}: n={s.height:>6} ({s.height/c.height*100:4.1f}%)  NDCG@10={s['ndcg'].mean():.4f}")
    return D, c

DS_R, co_r = cold_report("tgbn-reddit")

0it [00:00, ?it/s]

96354it [00:00, 963393.92it/s]

204270it [00:00, 1031451.78it/s]

307416it [00:00, 1022347.68it/s]

420074it [00:00, 1063270.86it/s]

...

11856421it [00:11, 1022539.06it/s]

11960215it [00:11, 1027083.61it/s]

12062946it [00:11, 1018632.23it/s]

12077151it [00:11, 1058794.95it/s]


=== tgbn-reddit ===  test (user,день)=517845 | overall NDCG@10=0.5597
  warmth==0 (НЕТ истории): n=141
  warmth<   10: n=   176 ( 0.0%)  NDCG@10=0.0946
  warmth<   50: n=   332 ( 0.1%)  NDCG@10=0.3044
  warmth<  200: n=  1139 ( 0.2%)  NDCG@10=0.5609
  warmth< 1000: n= 71959 (13.9%)  NDCG@10=0.5880


In [6]:
DS_K, co_k = cold_report("tgbn-token")

0it [00:00, ?it/s]

71361it [00:00, 713517.12it/s]

142713it [00:00, 687898.69it/s]

211566it [00:00, 666949.74it/s]

278331it [00:00, 655986.92it/s]

...

5207637it [00:08, 606060.58it/s]

5273825it [00:08, 622511.73it/s]

5340156it [00:09, 634596.55it/s]

5397704it [00:09, 592414.23it/s]


=== tgbn-token ===  test (user,день)=250550 | overall NDCG@10=0.4448
  warmth==0 (НЕТ истории): n=4221
  warmth<   10: n=  6971 ( 2.8%)  NDCG@10=0.1368
  warmth<   50: n= 14506 ( 5.8%)  NDCG@10=0.2860
  warmth<  200: n=163780 (65.4%)  NDCG@10=0.3668
  warmth< 1000: n=228434 (91.2%)  NDCG@10=0.4132


In [7]:
# --- Диагностика 2: какая ОСЬ «холодности» реально объясняет провал на genre? ---
# warmth = всего взаимодействий юзера (cnt[u])  vs  support = #ненулевых item'ов в строке M[u,:]
# Рычаг асимметрии: item'ов всего 513 -> у активного юзера строка почти плотная.
def stream_rich(DS, mem, cnt, loader, collect=False):
    ds = DS["ds"]; label_t = ds.get_label_time(); recs = []
    for batch in loader:
        sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
        if float(batch.t[-1]) > label_t:
            lt = ds.get_node_label(batch.t[-1])
            if lt is None: break
            lab_ts0 = float(lt[0][0]); ls, labs = lt[1].numpy(), lt[2].numpy()
            label_t = ds.get_label_time(); pm = tn < lab_ts0
            mem.update(sn[pm], dn[pm], tn[pm], wn[pm]); np.add.at(cnt, sn[pm], 1)
            if collect:
                pred = mem.read(ls, lab_ts0)
                supp = (pred > 0).sum(1)                      # сколько item'ов «горят» в строке
                for i in range(len(ls)):
                    if labs[i].sum() > 0:
                        recs.append((int(cnt[ls[i]]), int(supp[i]),
                                     float(ndcg_score(labs[i:i+1], pred[i:i+1], k=10))))
            sn, dn, tn, wn = sn[~pm], dn[~pm], tn[~pm], wn[~pm]
        mem.update(sn, dn, tn, wn); np.add.at(cnt, sn, 1)
    return recs

mem = HawkesMemory(DS["num_nodes"], DS["num_classes"], alpha, rphi); cnt = np.zeros(DS["num_nodes"])
stream_rich(DS, mem, cnt, DS["loaders"]["train"]); stream_rich(DS, mem, cnt, DS["loaders"]["val"])
rich = pl.DataFrame(stream_rich(DS, mem, cnt, DS["loaders"]["test"], collect=True),
                    schema=["warmth", "support", "ndcg"], orient="row")
DS["ds"].reset_label_time()
print(f"genre: n={rich.height}  overall={rich['ndcg'].mean():.4f}  "
      f"support строки: min={rich['support'].min()} med={int(rich['support'].median())} "
      f"max={rich['support'].max()}  (всего item'ов={DS['num_classes']})")
print("\nNDCG@10 по квинтилям SUPPORT (#горящих item'ов в строке):")
print(rich.with_columns(pl.col("support").qcut(5, allow_duplicates=True).alias("b"))
      .group_by("b").agg(pl.col("support").median().alias("med_support"),
                         pl.col("warmth").median().alias("med_warmth"),
                         pl.col("ndcg").mean().round(4).alias("ndcg"), pl.len().alias("n")).sort("med_support"))
print("\nКорреляция Спирмена ndcg~warmth vs ndcg~support:")
print(f"  ndcg~warmth  : {rich.select(pl.corr('warmth','ndcg',method='spearman')).item():.4f}")
print(f"  ndcg~support : {rich.select(pl.corr('support','ndcg',method='spearman')).item():.4f}")

genre: n=32151  overall=0.5203  support строки: min=1 med=154 max=395  (всего item'ов=513)

NDCG@10 по квинтилям SUPPORT (#горящих item'ов в строке):
shape: (5, 5)
┌─────────────┬─────────────┬────────────┬────────┬──────┐
│ b           ┆ med_support ┆ med_warmth ┆ ndcg   ┆ n    │
│ ---         ┆ ---         ┆ ---        ┆ ---    ┆ ---  │
│ cat         ┆ f64         ┆ f64        ┆ f64    ┆ u32  │
╞═════════════╪═════════════╪════════════╪════════╪══════╡
│ (-inf, 109] ┆ 84.0        ┆ 5056.0     ┆ 0.5417 ┆ 6681 │
│ (109, 135]  ┆ 124.0       ┆ 14707.0    ┆ 0.5396 ┆ 6411 │
│ (135, 173]  ┆ 154.0       ┆ 22236.0    ┆ 0.4968 ┆ 6204 │
│ (173, 220]  ┆ 190.0       ┆ 28576.5    ┆ 0.517  ┆ 6502 │
│ (220, inf]  ┆ 260.0       ┆ 40612.0    ┆ 0.5047 ┆ 6353 │
└─────────────┴─────────────┴────────────┴────────┴──────┘

Корреляция Спирмена ndcg~warmth vs ndcg~support:
  ndcg~warmth  : 0.1543
  ndcg~support : -0.0488


## Диагностика 3 · Разложение потерь: ось — **coverage**, а не warmth/support

Юзер-day предсказывается ранжированием строки `M[u,:]`. Диагностика 2 показала, что **support строки не объясняет NDCG** (корр ≈ 0). Правильная ось холодности — **покрытие позитивов**: попадают ли item'ы, с которыми юзер реально провзаимодействует *в этот день*, в support его строки.

Раскладываем NDCG каждого `(user, day)` по четырём **оракулам** (все каузальные, считаются по отдельности — это headroom-**измерения**, НЕ предсказатели; ничего не смешивается на выходе ⇒ запрет ансамблей соблюдён):

| оракул | предсказание | смысл |
|---|---|---|
| `floor` | строка из единиц (tie-averaged) | что даёт **пустая** память |
| `M` | наша M-only память | факт |
| `ceil` | истинные метки, занулённые вне виденных item'ов | **потолок при текущем покрытии** |
| `pop` | каузальная train-популярность item'ов | always-warm item-side прайор |

Тождество `floor ≤ M ≤ ceil ≤ 1`. Зазоры: `signal = M − floor`, `ranking_gap = ceil − M`, `coverage_gap = 1 − ceil`. `coverage = n_pos_seen / n_pos`.

- Холодная когорта проседает из-за **низкого ceil** ⇒ проблема **COVERAGE** (чинится только item-side достройкой `M·A`, frozen-from-train — рычаг `items ≪ users`).
- `ceil` высокий, но `M ≪ ceil` ⇒ проблема **RANKING** (чинится decay / write-transform, item-side тут ни при чём).

In [11]:
# Единый method-нейтральный коллектор: на том же строго-каузальном потоке считает
# per-(user,day) [warmth, seen_items, coverage] + 4 оракула NDCG (floor/M/ceil/pop).
def make_qpop(DS):  # каузальная train-популярность item'ов (message-space), нормирована
    tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    q = np.bincount(tr.dst.numpy().astype(np.int64), minlength=DS["num_classes"]).astype(np.float64)
    return q / max(q.sum(), 1.0)

def stream_diag(DS, mem, cnt, Cuv, q, loader, collect=False):
    ds = DS["ds"]; label_t = ds.get_label_time(); C = DS["num_classes"]; recs = []
    ones = np.ones((1, C)); qrow = q[None, :]
    for batch in loader:
        sn, dn, tn = batch.src.numpy(), batch.dst.numpy(), batch.t.numpy(); wn = batch.msg[:, 0].numpy()
        if float(batch.t[-1]) > label_t:
            lt = ds.get_node_label(batch.t[-1])
            if lt is None: break
            lab_ts0 = float(lt[0][0]); ls, labs = lt[1].numpy(), lt[2].numpy()
            label_t = ds.get_label_time(); pm = tn < lab_ts0
            mem.update(sn[pm], dn[pm], tn[pm], wn[pm]); np.add.at(cnt, sn[pm], 1)
            np.add.at(Cuv, (sn[pm].astype(np.int64), dn[pm].astype(np.int64)), 1)
            if collect:
                pred = mem.read(ls, lab_ts0)
                for i in range(len(ls)):
                    yi = labs[i]
                    if yi.sum() > 0:
                        u = int(ls[i]); seen = Cuv[u] > 0; pos = yi > 0
                        n_pos = int(pos.sum()); n_pos_seen = int((seen & pos).sum())
                        Y = yi[None]
                        recs.append((float(lab_ts0), u, int(cnt[u]), int(seen.sum()), n_pos, n_pos_seen,
                                     n_pos_seen / n_pos,
                                     float(ndcg_score(Y, pred[i:i+1], k=10)),       # M
                                     float(ndcg_score(Y, ones, k=10)),              # floor
                                     float(ndcg_score(Y, (yi * seen)[None], k=10)), # ceil (within-support)
                                     float(ndcg_score(Y, qrow, k=10))))             # pop
            sn, dn, tn, wn = sn[~pm], dn[~pm], tn[~pm], wn[~pm]
        mem.update(sn, dn, tn, wn); np.add.at(cnt, sn, 1)
        np.add.at(Cuv, (sn.astype(np.int64), dn.astype(np.int64)), 1)
    return recs

def run_diag(DS, phi, alpha, q):
    N, C = DS["num_nodes"], DS["num_classes"]
    mem = HawkesMemory(N, C, alpha, phi); cnt = np.zeros(N); Cuv = np.zeros((N, C), dtype=np.int32)
    stream_diag(DS, mem, cnt, Cuv, q, DS["loaders"]["train"])
    stream_diag(DS, mem, cnt, Cuv, q, DS["loaders"]["val"])
    recs = stream_diag(DS, mem, cnt, Cuv, q, DS["loaders"]["test"], collect=True)
    DS["ds"].reset_label_time()
    return pl.DataFrame(recs, schema=["ts","user","warmth","seen_items","n_pos","n_pos_seen","coverage",
                                      "ndcg","floor","ceil","pop"], orient="row")

q_g = make_qpop(DS)
dg = run_diag(DS, rphi, alpha, q_g)
# Инвариант: M<=ceil (ceil — оптимум внутри support). floor НЕ нижняя граница по строкам (это когортный референс).
assert (dg["ndcg"] <= dg["ceil"] + 1e-9).all(), "M>ceil — баг в оракуле ceil"
assert (dg["seen_items"] <= dg["warmth"]).all() and dg["coverage"].is_between(0, 1).all()
print(f"genre diag: n={dg.height}  ПАРИТЕТ ndcg={dg['ndcg'].mean():.4f} (cell2=0.5203)  "
      f"q.sum={q_g.sum():.3f} max(q)={q_g.max():.3f}  (строк floor>M: {(dg['floor']>dg['ndcg']).mean()*100:.1f}%)")
print("Средние оракулы:  floor={:.4f}  M={:.4f}  ceil={:.4f}  pop={:.4f}".format(
    dg["floor"].mean(), dg["ndcg"].mean(), dg["ceil"].mean(), dg["pop"].mean()))
print("Зазоры:           signal=M-floor={:+.4f}  ranking_gap=ceil-M={:+.4f}  coverage_gap=1-ceil={:+.4f}".format(
    dg["ndcg"].mean()-dg["floor"].mean(), dg["ceil"].mean()-dg["ndcg"].mean(), 1-dg["ceil"].mean()))
print(f"coverage: med={dg['coverage'].median():.3f}  =1 (все позитивы видены): {(dg['coverage']==1).mean()*100:.1f}%  "
      f"=0: {(dg['coverage']==0).mean()*100:.1f}%  |  pop>floor (item-side headroom): {dg['pop'].mean()-dg['floor'].mean():+.4f}")

genre diag: n=32151  ПАРИТЕТ ndcg=0.5203 (cell2=0.5203)  q.sum=1.000 max(q)=0.123  (строк floor>M: 7.1%)
Средние оракулы:  floor=0.0138  M=0.5203  ceil=0.9867  pop=0.3547
Зазоры:           signal=M-floor=+0.5065  ranking_gap=ceil-M=+0.4664  coverage_gap=1-ceil=+0.0133
coverage: med=1.000  =1 (все позитивы видены): 82.7%  =0: 0.2%  |  pop>floor (item-side headroom): +0.3409


In [12]:
# genre: разложение по warmth-когортам + по coverage-срезам — где теряется NDCG?
dgc = (dg.with_columns(pl.col("warmth").qcut(5, allow_duplicates=True).alias("bin"))
       .group_by("bin").agg(
           pl.col("warmth").median().cast(pl.Int64).alias("med_w"),
           pl.col("floor").mean().round(4).alias("floor"), pl.col("ndcg").mean().round(4).alias("M"),
           pl.col("ceil").mean().round(4).alias("ceil"), pl.col("pop").mean().round(4).alias("pop"),
           pl.col("coverage").mean().round(3).alias("cov"),
           (pl.col("ceil") - pl.col("ndcg")).mean().round(4).alias("rank_gap"),
           (1 - pl.col("ceil")).mean().round(4).alias("cov_gap"), pl.len().alias("n"))
       .sort("med_w"))
print("genre · разложение по warmth-когортам (холодные→тёплые):")
print(dgc)
print("\ngenre · по coverage-срезам (доля сегодняшних позитивов в support строки):")
for lab, sub in [("coverage==1", dg.filter(pl.col("coverage") == 1)),
                 ("0<cov<1", dg.filter((pl.col("coverage") > 0) & (pl.col("coverage") < 1))),
                 ("coverage==0", dg.filter(pl.col("coverage") == 0))]:
    if sub.height:
        print(f"  {lab:12} n={sub.height:6} ({sub.height/dg.height*100:5.1f}%)  "
              f"M={sub['ndcg'].mean():.4f}  ceil={sub['ceil'].mean():.4f}  "
              f"pop={sub['pop'].mean():.4f}  floor={sub['floor'].mean():.4f}")

genre · разложение по warmth-когортам (холодные→тёплые):
shape: (5, 10)
┌────────────────┬───────┬────────┬────────┬───┬───────┬──────────┬─────────┬──────┐
│ bin            ┆ med_w ┆ floor  ┆ M      ┆ … ┆ cov   ┆ rank_gap ┆ cov_gap ┆ n    │
│ ---            ┆ ---   ┆ ---    ┆ ---    ┆   ┆ ---   ┆ ---      ┆ ---     ┆ ---  │
│ cat            ┆ i64   ┆ f64    ┆ f64    ┆   ┆ f64   ┆ f64      ┆ f64     ┆ u32  │
╞════════════════╪═══════╪════════╪════════╪═══╪═══════╪══════════╪═════════╪══════╡
│ (-inf, 6715]   ┆ 3259  ┆ 0.0132 ┆ 0.4862 ┆ … ┆ 0.913 ┆ 0.4726   ┆ 0.0412  ┆ 6431 │
│ (6715, 15388]  ┆ 10596 ┆ 0.0134 ┆ 0.4779 ┆ … ┆ 0.975 ┆ 0.5115   ┆ 0.0106  ┆ 6430 │
│ (15388, 26316] ┆ 20438 ┆ 0.0136 ┆ 0.5074 ┆ … ┆ 0.983 ┆ 0.4851   ┆ 0.0075  ┆ 6431 │
│ (26316, 45969] ┆ 35218 ┆ 0.0137 ┆ 0.5274 ┆ … ┆ 0.988 ┆ 0.4684   ┆ 0.0042  ┆ 6429 │
│ (45969, inf]   ┆ 69495 ┆ 0.0151 ┆ 0.6027 ┆ … ┆ 0.99  ┆ 0.3944   ┆ 0.003   ┆ 6430 │
└────────────────┴───────┴────────┴────────┴───┴───────┴──────────┴─────────┴─

In [13]:
# reddit: то же разложение, фокус на ИСТИННО холодных (141 нулевой юзер). DS_R уже в ядре — без перезагрузки.
rphi_r = make_rank_phi(DS_R); q_r = make_qpop(DS_R)
dr = run_diag(DS_R, rphi_r, alpha, q_r)   # alpha = log2/(25*86400), как в cell2/4
print(f"reddit diag: n={dr.height}  ПАРИТЕТ M={dr['ndcg'].mean():.4f} (cell4=0.5597)  "
      f"floor={dr['floor'].mean():.4f} ceil={dr['ceil'].mean():.4f} pop={dr['pop'].mean():.4f}  |  "
      f"cov=1: {(dr['coverage']==1).mean()*100:.1f}%  cov=0: {(dr['coverage']==0).mean()*100:.1f}%")
print("\nreddit · по warmth-порогам  (M / ceil / pop / floor / cov):")
for thr in [1, 10, 50, 200, 10**18]:
    s = dr.filter(pl.col("warmth") < thr)
    tag = f"warmth<{thr}" if thr < 10**18 else "ALL"
    if s.height:
        print(f"  {tag:12} n={s.height:6} ({s.height/dr.height*100:4.1f}%)  M={s['ndcg'].mean():.4f}  "
              f"ceil={s['ceil'].mean():.4f}  pop={s['pop'].mean():.4f}  floor={s['floor'].mean():.4f}  "
              f"cov={s['coverage'].mean():.3f}  | cov_gap=1-ceil={1-s['ceil'].mean():.4f}")
s0 = dr.filter(pl.col("warmth") == 0)
print(f"\nИстинно холодные warmth==0 (n={s0.height}): M={s0['ndcg'].mean():.4f}  ceil={s0['ceil'].mean():.4f}  "
      f"pop={s0['pop'].mean():.4f}  floor={s0['floor'].mean():.4f}  →  item-side headroom pop-floor="
      f"{s0['pop'].mean()-s0['floor'].mean():+.4f}  (D4-гейт: >0.05?)")

reddit diag: n=517845  ПАРИТЕТ M=0.5597 (cell4=0.5597)  floor=0.0075 ceil=0.9706 pop=0.3004  |  cov=1: 89.5%  cov=0: 1.1%

reddit · по warmth-порогам  (M / ceil / pop / floor / cov):
  warmth<1     n=   141 ( 0.0%)  M=0.0081  ceil=0.0081  pop=0.5138  floor=0.0081  cov=0.000  | cov_gap=1-ceil=0.9919
  warmth<10    n=   176 ( 0.0%)  M=0.0946  ceil=0.0998  pop=0.4981  floor=0.0081  cov=0.059  | cov_gap=1-ceil=0.9002
  warmth<50    n=   332 ( 0.1%)  M=0.3044  ceil=0.3902  pop=0.4706  floor=0.0078  cov=0.311  | cov_gap=1-ceil=0.6098
  warmth<200   n=  1139 ( 0.2%)  M=0.5609  ceil=0.7519  pop=0.4537  floor=0.0079  cov=0.686  | cov_gap=1-ceil=0.2481
  ALL          n=517845 (100.0%)  M=0.5597  ceil=0.9706  pop=0.3004  floor=0.0075  cov=0.963  | cov_gap=1-ceil=0.0294

Истинно холодные warmth==0 (n=141): M=0.0081  ceil=0.0081  pop=0.5138  floor=0.0081  →  item-side headroom pop-floor=+0.5056  (D4-гейт: >0.05?)


In [14]:
# token: то же разложение — единственный датасет с БОЛЬШИМ холодным срезом (cell5: 65% warmth<200). DS_K уже в ядре.
rphi_k = make_rank_phi(DS_K); q_k = make_qpop(DS_K)
dk = run_diag(DS_K, rphi_k, alpha, q_k)
print(f"token diag: n={dk.height}  ПАРИТЕТ M={dk['ndcg'].mean():.4f} (cell5=0.4448)  "
      f"floor={dk['floor'].mean():.4f} ceil={dk['ceil'].mean():.4f} pop={dk['pop'].mean():.4f}  |  "
      f"cov=1: {(dk['coverage']==1).mean()*100:.1f}%  cov=0: {(dk['coverage']==0).mean()*100:.1f}%")
print("\ntoken · по warmth-порогам  (M / ceil / pop / floor / cov / cov_gap):")
for thr in [1, 10, 50, 200, 1000, 10**18]:
    s = dk.filter(pl.col("warmth") < thr)
    tag = f"warmth<{thr}" if thr < 10**18 else "ALL"
    if s.height:
        print(f"  {tag:12} n={s.height:7} ({s.height/dk.height*100:5.1f}%)  M={s['ndcg'].mean():.4f}  "
              f"ceil={s['ceil'].mean():.4f}  pop={s['pop'].mean():.4f}  floor={s['floor'].mean():.4f}  "
              f"cov={s['coverage'].mean():.3f}  cov_gap={1-s['ceil'].mean():.4f}")
# Сколько overall можно вернуть, заменив M на max(M,pop) на холодных (грубая оценка headroom-а сверху)
gain = (dk.select((pl.max_horizontal('pop','ndcg') - pl.col('ndcg')).mean()).item())
s0 = dk.filter(pl.col("warmth") == 0)
print(f"\ntoken warmth==0 (n={s0.height}): M={s0['ndcg'].mean():.4f} pop={s0['pop'].mean():.4f} "
      f"floor={s0['floor'].mean():.4f} → headroom pop-floor={s0['pop'].mean()-s0['floor'].mean():+.4f}")
print(f"ОЦЕНКА сверху: заменить M→pop там где pop>M поднимает overall на ≈ {gain:+.4f}  (0.4448 → ~{0.4448+gain:.4f})")

token diag: n=250550  ПАРИТЕТ M=0.4448 (cell5=0.4448)  floor=0.0046 ceil=0.6269 pop=0.0689  |  cov=1: 55.7%  cov=0: 33.2%

token · по warmth-порогам  (M / ceil / pop / floor / cov / cov_gap):
  warmth<1     n=   4221 (  1.7%)  M=0.0046  ceil=0.0046  pop=0.0374  floor=0.0046  cov=0.000  cov_gap=0.9954
  warmth<10    n=   6971 (  2.8%)  M=0.1368  ceil=0.1462  pop=0.0386  floor=0.0046  cov=0.137  cov_gap=0.8538
  warmth<50    n=  14506 (  5.8%)  M=0.2860  ceil=0.3386  pop=0.0455  floor=0.0046  cov=0.329  cov_gap=0.6614
  warmth<200   n= 163780 ( 65.4%)  M=0.3668  ceil=0.5458  pop=0.0494  floor=0.0046  cov=0.536  cov_gap=0.4542
  warmth<1000  n= 228434 ( 91.2%)  M=0.4132  ceil=0.5987  pop=0.0611  floor=0.0046  cov=0.589  cov_gap=0.4013
  ALL          n= 250550 (100.0%)  M=0.4448  ceil=0.6269  pop=0.0689  floor=0.0046  cov=0.618  cov_gap=0.3731

token warmth==0 (n=4221): M=0.0046 pop=0.0374 floor=0.0046 → headroom pop-floor=+0.0328
ОЦЕНКА сверху: заменить M→pop там где pop>M поднимает overa

In [15]:
# Кросс-датасетное разложение потерь NDCG@10 до 1.0 — где живёт headroom?
import plotly.graph_objects as go
names = ["genre", "reddit", "token"]; D = {"genre": dg, "reddit": dr, "token": dk}
M  = [D[n]["ndcg"].mean() for n in names]
rg = [(D[n]["ceil"] - D[n]["ndcg"]).mean() for n in names]   # ranking_gap: чинится read/write-правилом
cg = [(1 - D[n]["ceil"]).mean() for n in names]              # coverage_gap: чинится ТОЛЬКО item-side достройкой
pop = [D[n]["pop"].mean() for n in names]
fig = go.Figure()
fig.add_bar(name="M (факт)", x=names, y=M, marker_color="#2c7fb8",
            text=[f"{v:.3f}" for v in M], textposition="inside")
fig.add_bar(name="ranking_gap = ceil−M (read/write-правило)", x=names, y=rg, marker_color="#fdae61",
            text=[f"{v:.3f}" for v in rg], textposition="inside")
fig.add_bar(name="coverage_gap = 1−ceil (item-side M·A)", x=names, y=cg, marker_color="#d7191c",
            text=[f"{v:.3f}" for v in cg], textposition="inside")
fig.add_scatter(name="pop-оракул (популярность)", x=names, y=pop, mode="markers+text",
                marker=dict(symbol="diamond", size=16, color="black"),
                text=[f"{v:.3f}" for v in pop], textposition="top center")
fig.update_layout(barmode="stack", height=460,
                  title="Разложение NDCG@10 до 1.0: M + ranking_gap + coverage_gap (◆ = популярность)",
                  yaxis_title="NDCG@10 (стопка = 1.0)", legend=dict(orientation="h", y=-0.18))
fig.show()

# token: NDCG vs coverage — покрытие жёстко ограничивает качество (где cold реально живёт)
tb = (dk.with_columns((pl.col("coverage") * 10).floor().clip(0, 9).alias("cb"))
        .group_by("cb").agg(pl.col("ndcg").mean().alias("M"), pl.col("ceil").mean().alias("ceil"),
                            pl.len().alias("n")).sort("cb"))
fig2 = px.line(tb.to_pandas(), x="cb", y=["M", "ceil"], markers=True,
               title="token: NDCG@10 vs дециль coverage (доля позитивов в support)",
               labels={"cb": "дециль coverage (0=ничего видно, 9=всё)", "value": "NDCG@10", "variable": ""})
fig2.show()
print("token NDCG по децилям coverage:"); print(tb)

token NDCG по децилям coverage:
shape: (10, 4)
┌─────┬──────────┬──────────┬────────┐
│ cb  ┆ M        ┆ ceil     ┆ n      │
│ --- ┆ ---      ┆ ---      ┆ ---    │
│ f64 ┆ f64      ┆ f64      ┆ u32    │
╞═════╪══════════╪══════════╪════════╡
│ 0.0 ┆ 0.000433 ┆ 0.004627 ┆ 83137  │
│ 1.0 ┆ 0.122221 ┆ 0.179403 ┆ 253    │
│ 2.0 ┆ 0.160968 ┆ 0.312754 ┆ 1226   │
│ 3.0 ┆ 0.217316 ┆ 0.414798 ┆ 3056   │
│ 4.0 ┆ 0.261714 ┆ 0.485237 ┆ 446    │
│ 5.0 ┆ 0.329022 ┆ 0.586142 ┆ 13702  │
│ 6.0 ┆ 0.398783 ┆ 0.713734 ┆ 4401   │
│ 7.0 ┆ 0.4373   ┆ 0.792175 ┆ 1726   │
│ 8.0 ┆ 0.454235 ┆ 0.84878  ┆ 1802   │
│ 9.0 ┆ 0.728446 ┆ 0.999388 ┆ 140801 │
└─────┴──────────┴──────────┴────────┘


## Диагностика · ИТОГ: кросс-датасетная матрица режимов отказа

Все три M-only паритетны с baseline (genre 0.5203, reddit 0.5597, token 0.4448 ✓); инвариант `M≤ceil` — 0 нарушений.

| датасет | M | ceil | pop | cov=0 | `signal`=M−floor | `ranking_gap`=ceil−M | `coverage_gap`=1−ceil |
|---|---|---|---|---|---|---|---|
| **genre** | 0.520 | 0.987 | 0.355 | 0.2% | 0.507 | **0.466** | 0.013 |
| **reddit** | 0.560 | 0.971 | 0.300 | 1.1% | 0.552 | **0.411** | 0.029 |
| **token** | 0.445 | 0.627 | 0.069 | 33.2% | 0.440 | 0.182 | **0.373** |

**Три вывода, гейтирующие направление:**

1. **Popularity-backoff (EB-метод) мёртв на всех датасетах.** Оракул `max(M,pop)` по строкам поднимает overall: genre ≈0, reddit ≈+0.0003, **token +0.0067**. На token популярность (0.069) хуже M даже на холодных (`warmth<10`: M=0.137 > pop=0.039). Отвергаем — и это та форма, что p3/backlog и так помечали как скрытый ансамбль.

2. **genre/reddit: headroom — это RANKING внутри support, а не cold-start.** ceil≈0.97–0.99 ⇒ нужные item'ы почти всегда в строке, теряем на их ранжировании (`ranking_gap` 0.41–0.47). Это домен read/write-правила (расширение p3), single-model. Холодные по warmth <1% ⇒ overall ими не двигается.

3. **token — единственный с большим `coverage_gap` (0.373; 33% строк без позитивов в support).** Закрыть можно ТОЛЬКО item-side co-occurrence достройкой `M·A` (популярность бесполезна) — это санкционированный рычаг T5/H3 и ровно асимметрия `items≪users`. Открыто: захватывает ли co-occurrence этот headroom (или позитивы непредсказуемы).

**Развилка для p4** (см. вопрос ниже): (a) дешёвый оракул-гейт «ловит ли `M·A` token-овый coverage_gap» → решает судьбу core-тезиса; (b) сразу строить item-side `M·A`-метод на token; (c) пивот на `ranking_gap` (genre, лучший single-model read-rule).